# Lab 11 - Static LLM vs Agentic System
**Section covered: 05 - Static vs Agentic**

## The goal
Run the **same task** two ways: static LLM call, then full agent with tools.
See what each can and cannot do. Live code. Real output.

## The task
> *What is the current weather in London and Tokyo? I have GBP 500 for flights - is that realistic? Convert GBP 500 to USD and EUR.*

A static LLM will hedge or guess. An agent uses tools and returns real answers.

## What you will learn
- Where the static LLM ceiling sits: no external data, no actions
- How tools and a loop break through that ceiling
- Why the architecture difference - not the model - is everything

## Prerequisites
- ANTHROPIC_API_KEY in your .env file
- anthropic and python-dotenv installed

In [ ]:
import anthropic, datetime
from dotenv import load_dotenv
load_dotenv()
client = anthropic.Anthropic()
print('Connected to Anthropic API')
print(f'Time: {datetime.datetime.now().strftime("%H:%M:%S")}')

## Approach A - Static LLM (single call, no tools)
The model answers from training data only. Watch for hedging on weather and flights.

In [ ]:
TASK = (
    "What is the current weather in London and Tokyo right now? "
    "I have a budget of GBP 500 for flights between them - is that realistic? "
    "Also convert GBP 500 to USD and EUR."
)

print("APPROACH A: STATIC LLM")
print("-" * 45)
r = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=600,
    system="You are a helpful assistant. Answer the question as best you can.",
    messages=[{"role": "user", "content": TASK}]
)
print(r.content[0].text)
print(f"\nAPI calls: 1  |  Tool calls: 0  |  Tokens: {r.usage.input_tokens}in, {r.usage.output_tokens}out")

## Pause Here

Read the static answer carefully:
- **Weather**: hedged or disclaimed - model cannot access live data
- **Flight prices**: rough guess from training, could be outdated
- **Currency**: accurate - calculation, not live data

The model is not broken. It is architecturally incapable of accessing the real world.
Now watch what changes when we add tools and a loop.

## Tool Definitions
Simulated here - swap for real APIs without changing agent code.

In [ ]:
def get_weather(city):
    data = {
        "london": "12C, overcast with drizzle, 82% humidity, wind 14mph",
        "tokyo":  "24C, partly cloudy, 65% humidity, wind 8mph"
    }
    return f"Weather in {city.title()}: {data.get(city.lower(), 'Data unavailable')}"

def get_flight_estimate(origin, destination, budget_gbp):
    routes = {
        ("london","tokyo"): {"min":380,"max":850,"typical":520},
        ("tokyo","london"): {"min":380,"max":850,"typical":520}
    }
    r = routes.get((origin.lower(), destination.lower()), {"min":300,"max":900,"typical":500})
    ok = "Yes, economy seats available at this budget" if budget_gbp >= r["min"] else "Unlikely at this budget"
    return (f"Flights {origin.title()} to {destination.title()}: "
            f"economy GBP{r['min']}-GBP{r['max']} (typical GBP{r['typical']}). "
            f"Budget GBP{budget_gbp}: {ok}.")

def convert_currency(amount, from_currency, to_currency):
    rates = {"GBP":1.0,"USD":1.27,"EUR":1.17,"JPY":190.5,"AUD":1.95}
    f, t = from_currency.upper(), to_currency.upper()
    if f not in rates or t not in rates:
        return f"Currency not found. Options: {list(rates.keys())}"
    return f"{amount} {f} = {(amount/rates[f])*rates[t]:.2f} {t}"

TOOLS = [
    {"name":"get_weather","description":"Get current weather for a city.",
     "input_schema":{"type":"object","properties":{"city":{"type":"string"}},"required":["city"]}},
    {"name":"get_flight_estimate","description":"Estimate flight costs with budget assessment.",
     "input_schema":{"type":"object","properties":{
         "origin":{"type":"string"},"destination":{"type":"string"},"budget_gbp":{"type":"number"}},
         "required":["origin","destination","budget_gbp"]}},
    {"name":"convert_currency","description":"Convert between GBP, USD, EUR, JPY, AUD.",
     "input_schema":{"type":"object","properties":{
         "amount":{"type":"number"},"from_currency":{"type":"string"},"to_currency":{"type":"string"}},
         "required":["amount","from_currency","to_currency"]}}
]
TOOL_FNS = {"get_weather":get_weather,"get_flight_estimate":get_flight_estimate,"convert_currency":convert_currency}
print("Tools ready: get_weather, get_flight_estimate, convert_currency")

## Approach B - The Agent
Same task. Same model. Tools added. Loop enabled. Watch the [TOOL] calls appear.

In [ ]:
def run_agent(task, tools, fns, max_turns=10):
    messages = [{"role":"user","content":task}]
    calls = []
    for _ in range(max_turns):
        r = client.messages.create(
            model="claude-sonnet-4-6", max_tokens=1024,
            system="You are a helpful assistant. Use ALL available tools. Never guess - use the tools.",
            tools=tools, messages=messages)
        messages.append({"role":"assistant","content":r.content})
        if r.stop_reason == "end_turn":
            answer = next((b.text for b in r.content if hasattr(b,"text")), "")
            return {"answer": answer, "calls": calls}
        results = []
        for b in r.content:
            if b.type == "tool_use":
                print(f"  [TOOL] {b.name}({b.input})")
                out = fns[b.name](**b.input)
                print(f"  [RESULT] {out[:80]}")
                calls.append(b.name)
                results.append({"type":"tool_result","tool_use_id":b.id,"content":out})
        messages.append({"role":"user","content":results})
    return {"answer": "Max turns reached", "calls": calls}

print("APPROACH B: AGENT WITH TOOLS")
print("-" * 45)
result = run_agent(TASK, TOOLS, TOOL_FNS)

In [ ]:
print("\nAgent Answer:")
print(result["answer"])

## Side-by-Side Summary

In [ ]:
print("=" * 55)
print(f"STATIC LLM:  1 API call | 0 tool calls | hedged/guessed answers")
print(f"AGENT:       {len(result['calls'])+1} API calls | {len(result['calls'])} tool calls | real tool-grounded answers")
print()
print("SAME MODEL. SAME TRAINING.")
print("DIFFERENT ARCHITECTURE = DIFFERENT CAPABILITY.")
print()
print("The loop and the tools are the ENTIRE difference.")

## What just happened?

The model is identical in both approaches. The architecture changed:
- **Approach A**: model generates one answer and stops
- **Approach B**: model decides, calls tool, reads result, decides again, until complete

**Teaching tip:** Run both cells live back-to-back. Students see the hedging in A and specific data in B. The architectural point lands immediately.

---
**Next:** Lab 12 - Amazon Bedrock Knowledge Bases. Build a production RAG agent using AWS.